# Phân tích doanh thu cửa hàng

Dữ liệu giả lập năm 2025. Mục tiêu: luyện SQLite, pandas và Seaborn. Chạy lần lượt các ô; dữ liệu và kết quả tạo lại được bằng seed 42.

In [ ]:
from pathlib import Path
import sqlite3
import sys
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "sql/schema.sql").exists(), "Mở notebook từ repo hoặc thư mục notebooks"
sys.path.insert(0, str(ROOT / "src"))
from prepare_data import main as prepare
if not (ROOT / "database/analysis.db").exists():
    prepare()
sns.set_theme(style="whitegrid")

## 1. Hiểu dữ liệu

Một dòng của view `sales` là một sản phẩm trong một đơn hoàn tất. Vì vậy đếm dòng không tương đương đếm đơn. SQLite có quan hệ giữa bốn bảng; view đã JOIN các bảng và loại đơn hủy.

In [ ]:
with sqlite3.connect(f"{(ROOT / 'database/analysis.db').as_uri()}?mode=ro", uri=True) as conn:
    sales = pd.read_sql_query("SELECT * FROM sales", conn, parse_dates=["order_date"])
sales.head()

## 2. Kiểm tra chất lượng

Xem ô thiếu, dòng trùng và tính hợp lệ của doanh thu. Những kiểm tra này cần được điều chỉnh khi thay dữ liệu thật.

In [ ]:
display(sales.isna().sum().to_frame("missing"))
assert not sales.duplicated(["order_id", "product_name"]).any()
assert (sales["quantity"] > 0).all()
assert (sales["revenue_vnd"] == sales["quantity"] * sales["unit_price_vnd"]).all()
print("Completed orders:", sales["order_id"].nunique())

## 3. Truy vấn doanh thu theo tháng

Hãy thử tự viết GROUP BY trước khi chạy ô dưới.

In [ ]:
query = """
SELECT strftime('%Y-%m', order_date) AS month,
       SUM(revenue_vnd) AS revenue_vnd,
       COUNT(DISTINCT order_id) AS completed_orders
FROM sales
GROUP BY month
ORDER BY month
"""
with sqlite3.connect(f"{(ROOT / 'database/analysis.db').as_uri()}?mode=ro", uri=True) as conn:
    monthly = pd.read_sql_query(query, conn)
assert monthly["revenue_vnd"].sum() == sales["revenue_vnd"].sum()
monthly

In [ ]:
plot_data = monthly.assign(revenue_million_vnd=monthly["revenue_vnd"] / 1e6)
fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=plot_data, x="month", y="revenue_million_vnd", marker="o", ax=ax)
ax.set(title="Monthly revenue — synthetic data", xlabel="Month", ylabel="Million VND")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 4. Giá trị đơn hàng

Gộp các dòng sản phẩm về cấp đơn trước khi tính mean, median hoặc vẽ phân phối.

In [ ]:
orders = sales.groupby("order_id", as_index=False).agg(total_vnd=("revenue_vnd", "sum"))
display(orders["total_vnd"].describe())
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(orders["total_vnd"] / 1e6, bins=25, ax=ax)
ax.set(title="Completed order values", xlabel="Million VND")
plt.show()

## 5. Tạo bộ kết quả hoàn chỉnh

Chạy script phân tích để xuất năm biểu đồ, ba bảng CSV và báo cáo. Sau đó đọc `docs/exercises.md` và bổ sung nhận xét của bạn.

In [ ]:
from analyze import main as analyze
analyze()

## 6. Tự viết kết luận

- Phát hiện 1 và số liệu minh chứng:
- Phát hiện 2 và số liệu minh chứng:
- Phát hiện 3 và số liệu minh chứng:
- Hai đề xuất cần kiểm chứng:
- Giới hạn: dữ liệu ngẫu nhiên, không có chi phí; doanh thu không phải lợi nhuận. Một năm dữ liệu không đủ xác nhận tính mùa vụ.